# Chronos-2 — Panama Electric Load Forecasting (S1–S4)
**Darekar, Kim, Saxena | University of Trier | RCS SS2026**



- **S1** — Univariate zero-shot baseline
- **S2** — Covariate-informed forecasting (Panama City weather + calendar)
- **S3a** — Cross-city learning (Tocumen / Santiago / David temperatures, already in the raw dataset)
- **S3b** — STL decomposition cross-learning (trend + seasonal(24h) + residual, forecast jointly)
- **S4** — Ensemble-based forecasting (10 weather realizations → averaged)


In [3]:
# Standard library and third-party imports
import sys, os
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline
from transformers.utils.logging import disable_progress_bar
from IPython.display import Image, display
from statsmodels.tsa.seasonal import STL

sys.path.insert(0, os.path.join(os.getcwd(), "src"))
warnings.filterwarnings("ignore")

from config          import RESULTS_DIR, DATA_DIR, MODEL_ID, TARGET_COLUMN, TIMESTAMP_COLUMN, WEATHER_VARS, N_DAYS
from data_loader     import (pan_load, pan_preprocessing, fetch_weather, add_calendar, apply_semantic_pca,
                             merge_context_with_covariates, extract_prediction_timeframe, merge_location_weather,
                             PAN_START_DATE)
from metrics         import calculate_metrics, compute_summary_statistics, calculate_weather_correlations
from tests           import diebold_mariano_test
from plots           import plot_metrics_boxplots, plot_distribution, plot_time_series, plot_correlation
from scenarios       import s1_predict, s2_predict, s1_evaluation

print("All imports OK")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

ModuleNotFoundError: No module named 'entsoe'

## 1 — Load and Preprocess Panama Data

In [ ]:
# LOAD & PREPROCESS DATA
pan_loaded_data = pan_load()  # Uses default continuous_dataset.csv
print("\n--- Panama (PAN) Electricity Load Dataset ---")
print(pan_loaded_data)
pan_preprocessing(pan_loaded_data)

pan_context_df = pd.read_csv(DATA_DIR / "processed" / "pan_context_df.csv")
print("\n--- Target Variable Summary Statistics ---")
deduplicated_pan_context_df = pan_context_df.drop_duplicates(subset=[TIMESTAMP_COLUMN])
print(compute_summary_statistics(deduplicated_pan_context_df, columns=[TARGET_COLUMN]))
plot_distribution(deduplicated_pan_context_df, columns=[TARGET_COLUMN])

## 2 — Load Chronos-2
Downloads ~700 MB on first run, cached locally after that.

In [ ]:
disable_progress_bar()

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,                     # "amazon/chronos-2"  (set in config.py)
    device_map = "cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype = torch.bfloat16, # use float32 if you get dtype errors on CPU
)

print(f"Loaded {MODEL_ID}")

---
## S1 — Univariate Baseline (zero-shot)
Chronos-2 on raw demand only. Reference point for all adaptations below.

In [ ]:
pan_context_df = pd.read_csv(DATA_DIR / "processed" / "pan_context_df.csv")
pan_ground_truth = np.load(DATA_DIR / "processed" / "pan_horizon_true.npy")

pan_pred = s1_predict(pipeline, pan_context_df, save_path = RESULTS_DIR / "pan_pred.csv")
pan_res = s1_evaluation(pan_pred, pan_ground_truth, save_path = RESULTS_DIR / "pan_res.csv")

print("--- Panama (PAN) Electricity Load Forecasting Accuracy Metrics (RMSE and MAPE): Summary Statistics ---")
print(compute_summary_statistics(pan_res, columns=['rmse', 'mape']))

print("\n--- Panama (PAN) Electricity Load Time Series Forecasting Plot ---")
plot_time_series(pan_pred, pan_ground_truth, RESULTS_DIR / "pan_time_series.png")

---
## S2 — Covariate-Informed Forecasting
Weather (Panama City proxy, PCA-reduced) + calendar features via `predict_df(future_df=...)`.

### 2.1 Build weather covariates

In [ ]:
display(Image(DATA_DIR / "Images/Panama_Map.png"))

Approximately 47% of Panama's population resides in the Panama City metropolitan area. Weather
observations from Panama City alone are used as a proxy for national weather conditions, validated against
David and Colón below.

In [ ]:
# Fetch weather data for Panama City, David, and Colón
pan_city_weather = fetch_weather(
    start_date = (pd.to_datetime(PAN_START_DATE) - pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
    end_date = (pd.to_datetime(PAN_START_DATE) + pd.Timedelta(days=N_DAYS + 7 - 1)).strftime('%Y-%m-%d'),
    latitude = 8.9936, longitude = -79.5197, offset_hours = -5)

pan_dav_weather = fetch_weather(
    start_date = (pd.to_datetime(PAN_START_DATE) - pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
    end_date = (pd.to_datetime(PAN_START_DATE) + pd.Timedelta(days=N_DAYS + 7 - 1)).strftime('%Y-%m-%d'),
    latitude = 8.4273, longitude = -82.4309, offset_hours = -5)

pan_col_weather = fetch_weather(
    start_date = (pd.to_datetime(PAN_START_DATE) - pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
    end_date = (pd.to_datetime(PAN_START_DATE) + pd.Timedelta(days=N_DAYS + 7 - 1)).strftime('%Y-%m-%d'),
    latitude = 9.3603, longitude = -79.9002, offset_hours = -5)

pan_weather = merge_location_weather(pan_city_weather, pan_dav_weather, pan_col_weather, ('_city', '_dav', '_col'))
pan_weather.to_csv(DATA_DIR / "processed" / "pan_weather.csv", index=False)
print(f"Shape of merged Panama weather data: {pan_weather.shape}")

In [ ]:
pan_weather_corr = calculate_weather_correlations(pan_weather, suffixes=('_city', '_dav', '_col'), labels=('Panama_City', 'David', 'Colon'))
print("--- Weather Variable Correlations Across Panamanian Cities ---")
print(pan_weather_corr)
plot_correlation(pan_weather_corr, save_path = RESULTS_DIR / "pan_weather_corr.png")

### 2.2 Predict and evaluate

In [ ]:
# Panama City weather selected as the primary covariate source (per correlation analysis above).
pan_weather_single = pan_weather[['date'] + [col for col in pan_weather.columns if col.endswith('_city')]]
pan_weather_single.columns = [col.replace('_city', '') if col.endswith('_city') else col for col in pan_weather_single.columns]
pan_weather_reduced = apply_semantic_pca(pan_weather_single)

pan_covariates_df = add_calendar(pan_weather_reduced, country_code='PA')
# NOTE: which PCA covariates to drop should be tuned empirically the same way it was for AUS/LAT.
pan_covariates_df = pan_covariates_df.drop(columns=["dow"])
pan_covariates_df.to_csv(DATA_DIR / "processed" / "pan_covariates_df.csv", index=False)
print(pan_covariates_df)

In [ ]:
pan_prediction_timeframe = extract_prediction_timeframe(pan_context_df)

pan_cov_context_df = merge_context_with_covariates(pan_context_df, pan_covariates_df, save_name = "pan_cov_context_df.csv")
pan_future_df = merge_context_with_covariates(pan_prediction_timeframe, pan_covariates_df, save_name = "pan_future_df.csv")

pan_cov_pred = s2_predict(pipeline, pan_cov_context_df, pan_future_df, save_path = RESULTS_DIR / "pan_cov_pred.csv")
pan_cov_res = s1_evaluation(pan_cov_pred, pan_ground_truth, save_path = RESULTS_DIR / "pan_cov_res.csv")

print("--- Panama (PAN) S2 Accuracy Metrics (RMSE and MAPE): Summary Statistics ---")
print(compute_summary_statistics(pan_cov_res, columns=['rmse', 'mape']))

print("\n--- Panama (PAN) Diebold-Mariano Test: S2 vs S1 ---")
display(diebold_mariano_test(pan_ground_truth, pan_cov_pred, pan_pred))

---
## S3a — Cross-City Learning
Feed per-city temperature series (Tocumen, Santiago, David) as covariates alongside national demand.

Unlike S2, these are the **raw MERRA-2 reanalysis temperatures already embedded in `continuous_dataset.csv`**
(`T2M_toc`, `T2M_san`, `T2M_dav`) — no external API call needed, and no PCA reduction, so Chronos-2's group
attention sees all three cities directly instead of a single compressed component.

In [ ]:
# Pull the three raw per-city temperature series straight from the source dataset.
pan_raw = pan_load()  # same loader used in Section 1; returns the full continuous_dataset.csv frame
pan_raw = pan_raw.rename(columns={"datetime": "date"})
pan_raw["date"] = pd.to_datetime(pan_raw["date"])

pan_crosscity_weather = pan_raw[["date", "T2M_toc", "T2M_san", "T2M_dav"]].copy()

pan_crosscity_covariates_df = add_calendar(pan_crosscity_weather, country_code='PA')
pan_crosscity_covariates_df = pan_crosscity_covariates_df.drop(columns=["dow"])
pan_crosscity_covariates_df.to_csv(DATA_DIR / "processed" / "pan_crosscity_covariates_df.csv", index=False)
print(pan_crosscity_covariates_df)

In [ ]:
pan_crosscity_context_df = merge_context_with_covariates(pan_context_df, pan_crosscity_covariates_df, save_name = "pan_crosscity_context_df.csv")
pan_crosscity_future_df = merge_context_with_covariates(pan_prediction_timeframe, pan_crosscity_covariates_df, save_name = "pan_crosscity_future_df.csv")

pan_crosscity_pred = s2_predict(pipeline, pan_crosscity_context_df, pan_crosscity_future_df, save_path = RESULTS_DIR / "pan_crosscity_pred.csv")
pan_crosscity_res = s1_evaluation(pan_crosscity_pred, pan_ground_truth, save_path = RESULTS_DIR / "pan_crosscity_res.csv")

print("--- Panama (PAN) S3a Accuracy Metrics (RMSE and MAPE): Summary Statistics ---")
print(compute_summary_statistics(pan_crosscity_res, columns=['rmse', 'mape']))

print("\n--- Panama (PAN) Diebold-Mariano Test: S3a (cross-city) vs S1 (baseline) ---")
display(diebold_mariano_test(pan_ground_truth, pan_crosscity_pred, pan_pred))

print("\n--- Panama (PAN) Diebold-Mariano Test: S3a (cross-city) vs S2 (single-city PCA) ---")
display(diebold_mariano_test(pan_ground_truth, pan_crosscity_pred, pan_cov_pred))

---
## S3b — STL Decomposition Cross-Learning
Decompose demand → trend + seasonal (24h) + residual, forecast all three **jointly as separate series**
so Chronos-2's group attention can share structure across them, then sum the three component forecasts
back into a single demand forecast.

**ASSUMPTION:** this relies on `s1_predict` batching every `id` present in the input dataframe into one
`pipeline.predict_df()` call (as your S1 run already does for AUS, which processes ~180k rows / many
`AUS_SA_day_N` ids in a single call). That gives the model the *opportunity* to attend across the three
components of the same window, but whether it explicitly groups them (vs. treating all ids as
independent) depends on your `scenarios.py` implementation — check for a `group`/`item_id` grouping
argument in your `s1_predict` and `Chronos2Pipeline.predict_df` before trusting the cross-learning
interpretation of these numbers.

In [ ]:
# STL decomposition of the full Panama demand series (24h seasonal period = daily cycle).
pan_demand_series = pan_raw.set_index("date")[TARGET_COLUMN].asfreq("h").interpolate()
stl_result = STL(pan_demand_series, period=24, robust=True).fit()

pan_stl_components = pd.DataFrame({
    "date": pan_demand_series.index,
    "trend": stl_result.trend.values,
    "seasonal": stl_result.seasonal.values,
    "resid": stl_result.resid.values,
})
print(pan_stl_components.describe())

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
axes[0].plot(pan_stl_components["date"], pan_stl_components["trend"]);    axes[0].set_ylabel("Trend")
axes[1].plot(pan_stl_components["date"], pan_stl_components["seasonal"]); axes[1].set_ylabel("Seasonal (24h)")
axes[2].plot(pan_stl_components["date"], pan_stl_components["resid"]);    axes[2].set_ylabel("Residual")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "pan_stl_components.png")
plt.show()

In [ ]:
# Build one long-format dataframe per component, re-using pan_context_df's window/id scaffold
# but swapping in the STL component values in place of the raw target.
def build_component_context(base_context_df, component_series_df, component_name):
    merged = base_context_df.merge(
        component_series_df[["date", component_name]],
        left_on=TIMESTAMP_COLUMN, right_on="date", how="left"
    )
    merged[TARGET_COLUMN] = merged[component_name]
    merged["id"] = merged["id"].astype(str) + f"__{component_name}"
    return merged.drop(columns=["date", component_name])

pan_trend_context    = build_component_context(pan_context_df, pan_stl_components, "trend")
pan_seasonal_context = build_component_context(pan_context_df, pan_stl_components, "seasonal")
pan_resid_context    = build_component_context(pan_context_df, pan_stl_components, "resid")

pan_stl_context_df = pd.concat([pan_trend_context, pan_seasonal_context, pan_resid_context], ignore_index=True)
pan_stl_context_df.to_csv(DATA_DIR / "processed" / "pan_stl_context_df.csv", index=False)
print(f"Combined STL context shape: {pan_stl_context_df.shape}")
print(pan_stl_context_df.head())

In [ ]:
# Forecast all three components jointly in one predict_df call, then reconstruct demand = trend + seasonal + resid.
pan_stl_pred_raw = s1_predict(pipeline, pan_stl_context_df, save_path = RESULTS_DIR / "pan_stl_pred_raw.csv")

pan_stl_pred_raw["base_id"] = pan_stl_pred_raw["id"].str.replace(r"__(trend|seasonal|resid)$", "", regex=True)
pan_stl_pred = (
    pan_stl_pred_raw
    .groupby(["base_id", TIMESTAMP_COLUMN], as_index=False)["target"]
    .sum()
    .rename(columns={"base_id": "id"})
)
pan_stl_pred.to_csv(RESULTS_DIR / "pan_stl_pred.csv", index=False)

pan_stl_res = s1_evaluation(pan_stl_pred, pan_ground_truth, save_path = RESULTS_DIR / "pan_stl_res.csv")

print("--- Panama (PAN) S3b Accuracy Metrics (RMSE and MAPE): Summary Statistics ---")
print(compute_summary_statistics(pan_stl_res, columns=['rmse', 'mape']))

print("\n--- Panama (PAN) Diebold-Mariano Test: S3b (STL) vs S1 (baseline) ---")
display(diebold_mariano_test(pan_ground_truth, pan_stl_pred, pan_pred))

---
## S4 — Ensemble-Based Forecasting
10 weather realisations → average (mirrors Copernicus ERA5 EDA).

**Proper version** requires ERA5 Ensemble of Data Assimilations via the Copernicus CDS API (`cdsapi`
package + free registration at https://cds.climate.copernicus.eu/). I've stubbed that path below.
Since I can't reach that API from this environment, there's also a **synthetic fallback** so you can
verify the averaging mechanics end-to-end before your CDS credentials are set up — swap
`USE_SYNTHETIC_ENSEMBLE = False` once real ERA5 EDA data is wired in.

In [ ]:
USE_SYNTHETIC_ENSEMBLE = True   # set False once cdsapi / ERA5 EDA access is configured
N_ENSEMBLE_MEMBERS = 10

def fetch_era5_eda_ensemble(n_members=10):
    """ASSUMPTION / TODO: implement against Copernicus CDS ERA5 EDA (10 realizations, 3-hourly, 1x1deg).
    Requires: pip install cdsapi, and a ~/.cdsapirc with your API key.
    Should return a list of `n_members` weather DataFrames, each shaped like `pan_city_weather`
    (columns: date, temperature_2m, ...), one per ensemble realization, covering the same
    PAN_START_DATE .. PAN_START_DATE + N_DAYS + 7 window used elsewhere in this notebook.
    """
    raise NotImplementedError(
        "Wire this up to cdsapi + the Copernicus ERA5 EDA product once CDS credentials are available. "
        "See: https://cds.climate.copernicus.eu/"
    )

def synthetic_weather_ensemble(base_weather_df, n_members=10, temp_col="temperature_2m", noise_std_c=0.8):
    """Stand-in ensemble: perturbs the deterministic Panama City forecast with Gaussian noise on
    temperature only, roughly calibrated to typical short-range ERA5 EDA spread (~0.5-1.5C).
    NOT a substitute for real ensemble data — use only to test the S4 pipeline mechanics."""
    rng = np.random.default_rng(42)
    members = []
    for m in range(n_members):
        member_df = base_weather_df.copy()
        member_df[temp_col] = member_df[temp_col] + rng.normal(0, noise_std_c, size=len(member_df))
        members.append(member_df)
    return members

if USE_SYNTHETIC_ENSEMBLE:
    pan_weather_ensemble = synthetic_weather_ensemble(pan_weather_single, n_members=N_ENSEMBLE_MEMBERS)
    print(f"Generated {len(pan_weather_ensemble)} SYNTHETIC weather ensemble members (placeholder for ERA5 EDA).")
else:
    pan_weather_ensemble = fetch_era5_eda_ensemble(n_members=N_ENSEMBLE_MEMBERS)
    print(f"Fetched {len(pan_weather_ensemble)} ERA5 EDA weather ensemble members.")

In [ ]:
# Run S2-style covariate forecasting once per ensemble member, then average the resulting predictions.
pan_ensemble_preds = []

for m, member_weather in enumerate(pan_weather_ensemble):
    member_reduced = apply_semantic_pca(member_weather)
    member_covariates_df = add_calendar(member_reduced, country_code='PA').drop(columns=["dow"])

    member_context_df = merge_context_with_covariates(pan_context_df, member_covariates_df, save_name = f"pan_ens{m}_context_df.csv")
    member_future_df = merge_context_with_covariates(pan_prediction_timeframe, member_covariates_df, save_name = f"pan_ens{m}_future_df.csv")

    member_pred = s2_predict(pipeline, member_context_df, member_future_df, save_path = RESULTS_DIR / f"pan_ens{m}_pred.csv")
    pan_ensemble_preds.append(member_pred)
    print(f"Ensemble member {m+1}/{len(pan_weather_ensemble)} done.")

# Average across the 10 members (elementwise on the target column, aligned by id + timestamp).
pan_ens_pred = pan_ensemble_preds[0][["id", TIMESTAMP_COLUMN]].copy()
target_stack = np.stack([p["target"].values for p in pan_ensemble_preds], axis=0)
pan_ens_pred["target"] = target_stack.mean(axis=0)
pan_ens_pred.to_csv(RESULTS_DIR / "pan_ens_pred.csv", index=False)

pan_ens_res = s1_evaluation(pan_ens_pred, pan_ground_truth, save_path = RESULTS_DIR / "pan_ens_res.csv")

print("--- Panama (PAN) S4 Accuracy Metrics (RMSE and MAPE): Summary Statistics ---")
print(compute_summary_statistics(pan_ens_res, columns=['rmse', 'mape']))

print("\n--- Panama (PAN) Diebold-Mariano Test: S4 (ensemble) vs S1 (baseline) ---")
display(diebold_mariano_test(pan_ground_truth, pan_ens_pred, pan_pred))

---
## Results Summary — Panama, All Scenarios
Addresses **RQ1.1** (do adaptations improve on the univariate baseline?) and **RQ1.2** (which
configuration performs best) for the Panama dataset.

In [ ]:
pan_res_all = pd.concat([
    pan_res.assign(scenario="S1_baseline"),
    pan_cov_res.assign(scenario="S2_covariate"),
    pan_crosscity_res.assign(scenario="S3a_crosscity"),
    pan_stl_res.assign(scenario="S3b_stl"),
    pan_ens_res.assign(scenario="S4_ensemble"),
], ignore_index=True)

print("--- Panama (PAN) — Summary Statistics by Scenario ---")
for scenario, group in pan_res_all.groupby("scenario"):
    print(f"\n{scenario}:")
    print(compute_summary_statistics(group, columns=['rmse', 'mape']))

plot_metrics_boxplots(pan_res_all)